# 🎬 Movie Poster Genre Classifier – Prediction Notebook
This notebook loads the trained model saved during training and evaluates it on the held‑out test set. It also provides a helper to predict the genre of a single poster image.


In [ ]:
# Install required packages (only needed the first run)
!pip install -q tensorflow scikit-learn seaborn


In [ ]:
import os, json, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress TF warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Mount Google Drive (adjust if you run locally)
from google.colab import drive
drive.mount('/content/drive')

# Path where training notebook saved the model & assets
SAVE_DIR = '/content/drive/My Drive/movie_poster_v2'
MODEL_PATH = os.path.join(SAVE_DIR, 'best_model.keras')
CLASS_NAMES_PATH = os.path.join(SAVE_DIR, 'class_names.json')


In [ ]:
# ---- Custom LR schedule class (required for model loading) ----
class WarmUpCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=1e-7):
        super().__init__()
        self.initial_lr = float(initial_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps = float(total_steps)
        self.min_lr = float(min_lr)
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = self.initial_lr * (step / self.warmup_steps)
        cosine_steps = tf.maximum(step - self.warmup_steps, 0.0)
        cosine_total = self.total_steps - self.warmup_steps
        cosine = self.min_lr + 0.5 * (self.initial_lr - self.min_lr) * (1.0 + tf.cos(math.pi * cosine_steps / cosine_total))
        return tf.where(step < self.warmup_steps, warmup, cosine)
    def get_config(self):
        return {
            'initial_lr': self.initial_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps,
            'min_lr': self.min_lr
        }


In [ ]:
# Load the trained model
model = tf.keras.models.load_model(MODEL_PATH, custom_objects={'WarmUpCosineDecay': WarmUpCosineDecay})
model.summary(line_length=100)


In [ ]:
# Load class names (ensures correct ordering)
with open(CLASS_NAMES_PATH, 'r') as f:
    class_names = json.load(f)
NUM_CLASSES = len(class_names)
print('Classes:', class_names)


In [ ]:
# ---------- Prepare Test Dataset ----------
# These values must match the training notebook
IMG_SIZE = (300, 300)  # Same as training config
BATCH_SIZE = 16
SEED = 42
DATA_DIR = '/content/dataset/Movie_Posters_IMDb'  # Adjust if your data lives elsewhere

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.30,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)
# Split the held‑out portion into equal validation / test (as training notebook does)
total_held = test_ds_raw.cardinality().numpy()
val_batches = max(1, total_held // 2)
test_ds = test_ds_raw.skip(val_batches)

# Pre‑processing (no augmentation)
preprocess_fn = tf.keras.applications.efficientnet_v2.preprocess_input
def prepare_eval(images, labels):
    images = preprocess_fn(images)
    return images, labels
test_ds = test_ds.map(prepare_eval, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)


In [ ]:
# ---------- Evaluate on Test Set ----------
test_results = model.evaluate(test_ds, verbose=1)
metric_names = model.metrics_names
for name, val in zip(metric_names, test_results):
    print(f'{name:<25s}: {val:.4f}')


The above prints `accuracy` and `top3_accuracy` among other metrics, giving you the model's performance on unseen data.


In [ ]:
import io
from PIL import Image as PILImg
from google.colab import files

# ── Load model & class names ──────────────────────────────────
MODEL_PATH = os.path.join(SAVE_DIR, 'movie_poster_classifier_v2_final.keras')
NAMES_PATH = os.path.join(SAVE_DIR, 'class_names.json')

loaded_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'WarmUpCosineDecay': WarmUpCosineDecay}
)
with open(NAMES_PATH) as f:
    loaded_class_names = json.load(f)

print(f"Model loaded from: {MODEL_PATH}")
print(f"Classes: {loaded_class_names}")

# ── TTA augmentations ─────────────────────────────────────────
TTA_AUGS = [
    lambda x: x,                                        # original
    lambda x: tf.image.flip_left_right(x),              # h-flip
    lambda x: tf.image.adjust_brightness(x, 0.1),       # brighter
    lambda x: tf.image.adjust_brightness(x, -0.1),      # darker
    lambda x: tf.image.central_crop(x, 0.9),            # slight crop
]

def resize_fn(img, size=IMG_SIZE):
    return tf.image.resize(img, size)

def predict_with_tta(model, img_array_raw, n_augs=5):
    """
    img_array_raw: numpy array, shape (H, W, 3), dtype uint8, values 0-255
    Returns averaged probability vector.
    """
    img_tensor = tf.cast(img_array_raw, tf.float32)
    all_probs  = []

    augs_to_use = TTA_AUGS[:n_augs]
    for aug_fn in augs_to_use:
        augmented = aug_fn(img_tensor)
        resized   = resize_fn(augmented)
        # IMPORTANT: use the exact same preprocessing as training
        preprocessed = tf.keras.applications.efficientnet_v2.preprocess_input(resized)
        batch = tf.expand_dims(preprocessed, 0)
        probs = model.predict(batch, verbose=0)[0]
        all_probs.append(probs)

    averaged = np.mean(all_probs, axis=0)
    return averaged

# ── Upload & predict ──────────────────────────────────────────
print("\nPlease upload one or more movie poster images …")
uploaded = files.upload()

if not uploaded:
    print("No files uploaded.")
else:
    for filename, content in uploaded.items():
        print(f"\n{'='*55}")
        print(f"  File: {filename}")
        print(f"{'='*55}")

        try:
            pil_img = PILImg.open(io.BytesIO(content)).convert('RGB')
            img_np  = np.array(pil_img)                      # (H, W, 3) uint8

            # TTA prediction
            avg_probs = predict_with_tta(loaded_model, img_np, n_augs=5)
            top_idx   = int(np.argmax(avg_probs))
            top_label = loaded_class_names[top_idx]
            top_conf  = float(avg_probs[top_idx])

            # Display image + results
            fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 5))

            ax_img.imshow(pil_img)
            ax_img.set_title(f"Predicted: {top_label}  ({top_conf:.1%} confidence)",
                             fontsize=13, fontweight='bold',
                             color='green' if top_conf > 0.5 else 'orange')
            ax_img.axis('off')

            sorted_idx  = np.argsort(avg_probs)[::-1]
            sorted_probs = avg_probs[sorted_idx]
            sorted_names = [loaded_class_names[i] for i in sorted_idx]
            colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(sorted_names))]

            bars = ax_bar.barh(sorted_names[::-1], sorted_probs[::-1], color=colors[::-1])
            ax_bar.set_xlim(0, 1)
            ax_bar.set_xlabel('Confidence (TTA averaged)', fontsize=11)
            ax_bar.set_title('Genre Probability Distribution', fontsize=12, fontweight='bold')
            for bar, prob in zip(bars, sorted_probs[::-1]):
                ax_bar.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                            f'{prob:.1%}', va='center', fontsize=9)
            ax_bar.grid(axis='x', alpha=0.3)

            plt.suptitle(f'{filename}', fontsize=10, color='gray')
            plt.tight_layout()
            plt.show()

            print(f"  Top prediction : {top_label}  ({top_conf:.1%})")
            print("  Full distribution (TTA):")
            for i in sorted_idx:
                bar = '█' * int(avg_probs[i] * 30)
                print(f"    {loaded_class_names[i]:<12s}: {avg_probs[i]:.4f}  {bar}")

        except Exception as e:
            print(f"  [ERROR] Failed to process {filename}: {e}")
